In [ ]:
def class_acc(df, weight_class, attempted_col, landed_col, curr_date):
    """ get red and blue attempted and landed """

    df_weight_class = df[(df['weight_class'] == weight_class) & (df['date'] < curr_date)].dropna()
    
    class_acc_red = df_weight_class[landed_col + '_red'].sum() / df_weight_class[attempted_col + '_red'].sum()
    class_acc_blue = df_weight_class[landed_col + '_blue'].sum() / df_weight_class[attempted_col + '_blue'].sum()

    class_acc_mean = (class_acc_red + class_acc_blue) / 2

    return class_acc_mean

def expected_value_stats(df_, attempt_col, land_col):

    """" pre fight stats, per minute """

    df = df_.copy()

    red_col = []
    blue_col = []
    efficiency = defaultdict(list)
    fighter_history = defaultdict(lambda: defaultdict(list))

    def compute_fighter_performance(row, fighter_name, wc_acc, color):

        att_hist = [x for x in fighter_history[fighter_name]['attempted'] if pd.notna(x)]
        land_hist = [x for x in fighter_history[fighter_name]['landed'] if pd.notna(x)]

        if len(land_hist)!=0 and np.sum(att_hist)!=0 and not np.isnan(wc_acc):

            # get past accuracy of fighter, NOT INCLUDIG CURRENT
            acc = np.sum(land_hist) / np.sum(att_hist)

            # take most recent fight landed
            curr_fight_attempted = row[attempt_col + f'_{color}']

            # get expected strikes class
            expected_class = wc_acc * curr_fight_attempted

            # get expected fighter
            expected_fighter = acc * curr_fight_attempted

            # get performance
            performance = expected_fighter - expected_class

            if np.isnan(performance):
                print(wc_acc , curr_fight_attempted)

        else: 
            performance = None
        
        fighter_history[fighter_name]['attempted'].append(row[f'{attempt_col}_{color}'])
        fighter_history[fighter_name]['landed'].append(row[f'{land_col}_{color}'])

        return performance

    for _, row in df.iterrows(): 

        wc = row['weight_class']
        date = row['date']

        fighter_red = row['fighter_red']
        fighter_blue = row['fighter_blue']

        # get weighted average of weight class HISTORY
        wc_acc = class_acc(df, wc, attempt_col, land_col, date)

        # current fight performance 
        red_performance = compute_fighter_performance(row, fighter_red, wc_acc, 'red') 
        blue_performance = compute_fighter_performance(row, fighter_blue, wc_acc, 'blue') 

        # print(red_performance, blue_performance)

        # red_efficiency = time_decay_average(efficiency[fighter_red] ) if len(efficiency[fighter_red])!= 0 else None
        # blue_efficiency = time_decay_average(efficiency[fighter_blue]) if len(efficiency[fighter_blue])!= 0 else None

        # get performance average 
        red_efficiency = np.mean([x for x in efficiency[fighter_red] if x is not pd.isna(x)]) if len(efficiency[fighter_red])!= 0 else np.nan
        blue_efficiency = np.mean([x for x in efficiency[fighter_blue] if x is not pd.isna(x)]) if len(efficiency[fighter_blue])!= 0 else np.nan

        red_col.append(red_efficiency)
        blue_col.append(blue_efficiency)

        if red_performance is not None: 
            efficiency[fighter_red].append(red_performance)

        if blue_performance is not None:
            efficiency[fighter_blue].append(blue_performance)

    return np.column_stack([red_col, blue_col])

In [ ]:
def time_decay_average(arr, decay_lambda=0.13):
    arr = np.asarray(arr)
    
    # 0 = most recent, larger = older
    age = np.arange(len(arr)-1, -1, -1)
    
    weights = np.exp(-decay_lambda * age)

    weighted_avg =  np.sum(weights * arr) / np.sum(weights)
    if np.isnan(weighted_avg):
        print(arr)

    return weighted_avg


def opponent_avg_features(df, feat):

    """
    
    """
    red_col = []
    blue_col = []

    fighter_history = defaultdict(list)
    fighter_adjusted_history = defaultdict(lambda:[np.nan])

    for _, row in df.iterrows(): 

        fighter_red = row['fighter_red']
        fighter_blue = row['fighter_blue']

        red_col.append(fighter_adjusted_history[fighter_red][-1])
        blue_col.append(fighter_adjusted_history[fighter_blue][-1])

        red_feat = row[f'{feat}_red']
        blue_feat = row[f'{feat}_blue']

        if not pd.isna(blue_feat):
            fighter_history[fighter_red].append(blue_feat)

        if not pd.isna(red_feat):
            fighter_history[fighter_blue].append(red_feat)

        red_opp_hist = time_decay_average(fighter_history[fighter_red]) if len(fighter_history[fighter_red]) != 0 else np.nan
        blue_opp_hist = time_decay_average(fighter_history[fighter_blue]) if len(fighter_history[fighter_blue]) != 0 else np.nan

        fighter_adjusted_history[fighter_red].append(red_opp_hist)
        fighter_adjusted_history[fighter_blue].append(blue_opp_hist)

    return np.column_stack([red_col, blue_col])
        

In [ ]:
red_col, blue_col = expected_value_stats(df_combined, 'td_attempted_pm', 'td_landed_pm').T

In [ ]:
feat_red, feat_blue = opponent_avg_features(df_combined, 'age').T

In [ ]:
sns.histplot(feat_red)
plt.show()

sns.histplot(feat_blue)
plt.show()